# Module 6 — Day 2 Breakout: Build a Random Forest by Hand
## PHY 657, Spring 2026

---

Work with your partner. You have ~20 minutes.

The goal is to understand what `RandomForestClassifier` does by implementing it yourself from individual trees.

### Tasks

1. Create 5 bootstrap samples from the training data.
2. Train a decision tree on each bootstrap sample (with random feature subsets at each split).
3. Collect all 5 predictions for the test set and compute the majority vote.
4. Compare your hand-built ensemble to sklearn's `RandomForestClassifier`.

**Discussion question:** How often does the majority vote get a sample right when an individual tree gets it wrong? Why does this happen?

---
## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 14, 'figure.figsize': (8, 6)})
RNG = np.random.default_rng(42)

In [2]:
# Generate the dataset
X, y = make_moons(n_samples=400, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)
print(f'Train: {len(y_train)}, Test: {len(y_test)}')

Train: 280, Test: 120


---
## Task 1: Create bootstrap samples and train trees

A bootstrap sample is drawn **with replacement** from the training data, same size as the original.

The key setting is `max_features='sqrt'` — this makes each tree consider only a random subset of features at **every split**, which is what distinguishes a random forest tree from a plain decision tree. A different random subset is chosen at each split, so even two trees trained on the same data will make different decisions.

Fill in the loop below to create 5 bootstrap samples and train a tree on each.

In [ ]:
n_trees = 5
trees = []

for b in range(n_trees):
    # Step 1: draw bootstrap indices (with replacement)
    idx = RNG.choice(len(y_train), size=len(y_train), replace=True)
    X_boot, y_boot = X_train[idx], y_train[idx]
    
    # Step 2: train a tree on the bootstrap sample
    # YOUR CODE HERE — fill in the constructor
    tree = DecisionTreeClassifier()  # <-- what arguments should go here?
    tree.fit(X_boot, y_boot)
    trees.append(tree)
    
    # Report
    frac_unique = len(np.unique(idx)) / len(y_train)
    acc = tree.score(X_test, y_test)
    print(f'Tree {b}: {frac_unique:.1%} unique samples, test acc = {acc:.1%}')

print(f'\nTheory: ~{1 - 1/np.e:.1%} unique per bootstrap')

---
## Task 2: Majority vote

Collect all 5 trees' predictions for the test set, then take the majority vote.

For binary classification, the majority vote is: if the average prediction across trees is > 0.5, predict class 1; otherwise predict class 0.

In [ ]:
# Collect predictions: shape (n_trees, n_test)
all_preds = np.array([tree.predict(X_test) for tree in trees])

# YOUR CODE HERE: compute the majority vote
# Hint: average across trees (axis=0), then threshold at 0.5
majority_vote = ___

# Compare
ensemble_acc = accuracy_score(y_test, majority_vote)
individual_accs = [tree.score(X_test, y_test) for tree in trees]

print(f'Individual tree accuracies: {["{:.1%}".format(a) for a in individual_accs]}')
print(f'Mean individual accuracy:   {np.mean(individual_accs):.1%}')
print(f'Best individual accuracy:   {np.max(individual_accs):.1%}')
print(f'Ensemble (majority vote):   {ensemble_acc:.1%}')

---
## Task 3: When does the vote help?

The code below prints a table showing each tree's prediction alongside the vote and the true label for the first 20 test points. Run it and look at the cases where the vote disagrees with individual trees.

In [ ]:
# Print prediction table (provided — just run this)
print(f'{"Pt":>3s}  ', end='')
for b in range(n_trees):
    print(f'T{b} ', end='')
print(f'  Vote  True  Match?')
print('-' * 45)

for i in range(20):
    print(f'{i:>3d}   ', end='')
    for b in range(n_trees):
        print(f'{int(all_preds[b, i]):>2d} ', end='')
    vote_i = int(majority_vote[i])
    true_i = int(y_test[i])
    match = '  ✓' if vote_i == true_i else '  ✗'
    print(f'    {vote_i}     {true_i}   {match}')

# Count how often the vote saves us
vote_correct = (majority_vote == y_test)
any_tree_wrong = np.any(all_preds != y_test[np.newaxis, :], axis=0)
vote_saved = np.sum(vote_correct & any_tree_wrong)

print(f'\nOut of {len(y_test)} test points:')
print(f'  Vote correct when at least one tree was wrong: {vote_saved}')

---
## Task 4: Compare to sklearn

Verify that your hand-built ensemble gives similar results to `RandomForestClassifier`.

In [ ]:
# YOUR CODE HERE: fit a RandomForestClassifier with n_estimators=5
# and compare its test accuracy to your majority vote


# Bonus: try n_estimators=200 and see how much better it gets


---
## Discussion

**How often does the majority vote get a sample right when an individual tree gets it wrong?**

**Why does this happen?** Think about:
- Each tree sees a different bootstrap sample and considers different features at each split
- If one tree makes an error due to its particular training sample, do the others make the *same* error?
- How does this connect to averaging measurements in experimental physics?


